In [ ]:
from mnesis_boilerplate import torch, np, pd, plt, data_cache, figpath, RECOMPUTE, DEBUG, datetag, loss_fn, Params, SubplotParams
from mnesis_boilerplate import printfig, pprint, approx_equals, asdict
from mnesis_chains import StochasticSpikingPattern, HD_SNN
# RECOMPUTE = True
from mnesis_boilerplate import purge_unfinished_scans
purge_unfinished_scans(remove_locks=False, dry_run=False) # HACK

datetag = '2026-09-07'
Using device: mps


In [2]:
opt = Params()
opt

Params(datetag='2026-09-07', N_neuron=1024, num_delay=41, N_pattern=16, N_time=1000, N_pretime=50, p_A=0.00016, p_flip=0.01, seed=2026, lif_beta=0.7, lif_threshold=0.75, learn_beta=False, learn_threshold=False, do_pinv=True, do_deconv=True, num_epochs=256, num_warmup_epochs=16, base_lr=0.04, final_lr=0.0004, delta1=0.003, delta2=1e-05, dropout=0.6, alpha_surrogate=5.0, surrogate_name='FastSigmoid', loss_name='SpikeF1scoreLoss', reset_mechanism='zero', optimizer='adadelta', verbose=False, fig_width=15.92, phi=1.61803, fig_height=9.839125356142963, N_time_show=1000, N_neuron_show=1024, i_pattern=0, N_scan=14, N_cv=11)

In [3]:
N_cv = 10

In [4]:
# pprint('HACK - using previous computations for plotting purposes, but the code has changed since then. The outputs are not guaranteed to be correct.') 
# opt = Params()
# datetag = '2026-08-06'
# opt.datetag = datetag
# N_cv = 1
# opt

In [5]:
print(f'Spikes in one target {opt.p_A * opt.N_neuron * opt.N_time:.1f},  in a SM window {opt.p_A * opt.N_neuron * opt.num_delay:.1f}')

Spikes in one target 163.8,  in a SM window 6.7


In [6]:
print(f'for a value {opt.lif_beta=:.1f}, the time constant is {- 1 / np.log(opt.lif_beta):.1f} steps')

for a value opt.lif_beta=0.7, the time constant is 2.8 steps


## scanning parameters

In [7]:
pprint('Scanning core network parameters to evaluate learning performance and robustness')

Scanning core network parameters to evaluate learning performance and robustness


In [ ]:
import time
opt_scan = Params(datetag = datetag + '_scan',
                  # N_time = 2**14,
                #   fig_width = 3.25,
                  )

N_scanning = 13 // DEBUG

scan_dicts= {'N_pattern' :  np.linspace(1, 26, N_scanning, dtype=int), # [3, 5, 7, 11,  13, 17], #  , 34, 55, 89, 144, 233, 377, 610, 987, 1364, 2351
             'N_time' : np.geomspace(100, 2000, N_scanning, dtype=int)[:-1], #[2**k for k in range(6, 12)],
             'num_delay' : np.linspace(1, 60, N_scanning, dtype=int), # [2, 3, 5, 8, 13, 21, 34, 55, 89],
             'N_neuron' : np.geomspace(50, 2000, N_scanning, dtype=int), # [55, 89, 144, 233, 377, 610, 987, 1364], #13, 21, 34, 
             'p_A': np.geomspace(0.0002, 0.05, N_scanning, dtype=float) , # [0.0002, 0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05], #0.00001, 0.00002, 0.00005, 0.0001, .. , 0.1, 0.5
             'lif_threshold': np.linspace(0.6, 1.05, N_scanning, dtype=float), 
             'do_deconv' : [True, False],
             'do_pinv' : [True, False],
             'p_flip': np.geomspace(1e-5, 1e-2, N_scanning, dtype=float), 
            }
label_dicts= {'N_pattern' : 'number of patterns',
             'N_time' : 'Pattern duration',
             'num_delay' : 'Delay range',
             'N_neuron' : 'Presynaptic inputs',
             'p_A': 'Average firing',
             'lif_threshold': 'LIF threshold',
             'do_deconv' : 'Membrane deconvolution',
             'do_pinv' : 'Full pseudo-inverse computation',
              'p_flip': 'Bit flip probability',
            }


for key in scan_dicts:
    print(f"{50*'-'}\nscanning {key=}\n{50*'-'}", )
    filename = data_cache / f'{opt_scan.datetag}_{key}.json'
    lock_filename = filename.with_suffix('.lock')
    if RECOMPUTE : 
        filename.unlink(missing_ok=True) # FORCING RECOMPUTE
        lock_filename.unlink(missing_ok=True) # FORCING RECOMPUTE

    try:
        lock_filename.touch(exist_ok=False) # atomic (O_EXCL): fails if another process already holds this scan
    except FileExistsError:
        continue # running elsewhere, or a stale lock (see purge_unfinished_scans)
    try:
        # read the results only once the lock is held, so that they include the rows written by a previous holder
        if filename.exists():
            df_scan = pd.read_json(filename, orient='records')
        else:
            measure_columns = [key, 'i_cv', 'loss', 'time']
            df_scan = pd.DataFrame([], columns=measure_columns)
        # do print the progress if any item is not computed
        verb = any(df_scan[approx_equals(df_scan[key], value) & (df_scan["i_cv"] == i_cv)].empty
                   for value in scan_dicts[key] for i_cv in range(N_cv))
        for i_value, value in enumerate(np.random.permutation(scan_dicts[key])):
            # tolist() gives plain Python ints, so that the progress line reads i_cv=5 rather than i_cv=np.int64(5)
            for i_cv in np.random.permutation(N_cv).tolist():
                # Check if there's already a row with the same key value and i_cv
                mask = approx_equals(df_scan[key], value) & (df_scan["i_cv"] == i_cv)

                if df_scan[mask].empty :
                    if verb: print(f'|{i_cv=}| {i_value + 1}/{len(scan_dicts[key])}', end='\t')
                    # append a placeholder row with time zero to allow concurrent access to the file by other processes;
                    # read_json numbers the rows 0..n-1, so the label len(df_scan) is new and no computed row is overwritten
                    df_scan.loc[len(df_scan)] = {key:value, 'i_cv':i_cv, 'loss':float('inf'), 'time':0.0}
                    df_scan.to_json(filename, orient='records', indent=2)

                    # tic
                    since = time.time()
                    new_dict = asdict(opt_scan)
                    new_dict[key] = value
                    # the seed depends only on i_cv, not on the order of computation nor on the resumptions:
                    # each row is reproducible and a given i_cv shares the same motifs across the values of a key
                    new_dict['seed'] = opt_scan.seed + int(i_cv)
                    new_opt = Params(**new_dict)
                    new_hd = HD_SNN(new_opt, StochasticSpikingPattern())
         
                    new_hd.update_weight()
                    new_hd.learn_model(verbose=False)

                    try:
                        with torch.no_grad():
                            # seeded noisy cue, replay scored against the frozen motifs (as in notebooks 11-16);
                            # rows cached before 2026-10-26 were scored against an unseeded noisy realisation
                            target_ = new_hd.pattern_object(seed=new_opt.seed)
                            input_spikes = new_hd.get_input_spikes(target=target_, p_A=new_hd.opt.p_A, N_pretime=new_hd.opt.N_pretime, N_trigger_time=new_hd.opt.num_delay, N_time=new_hd.opt.N_time, seed=new_opt.seed)
                            _, _, spikes = new_hd.forward_pass(input_spikes)
                            spikes_ = spikes[:, :, (new_hd.opt.N_pretime+new_hd.opt.num_delay):(-new_hd.opt.N_pretime)]
                            loss_val = loss_fn(spikes_, new_hd.pattern_object.frozen_target[:, :, new_hd.opt.num_delay:]).item()
                    except Exception as e:
                        print(f"Error during evaluation: {e}")
                        loss_val = float('inf')  # Assign a large loss value to indicate failure
                        
                    # toc
                    elapsed_time = time.time() - since

                    df_scan = pd.read_json(filename, orient='records')
                    mask = approx_equals(df_scan[key], value) & (df_scan["i_cv"] == i_cv)
                    df_scan[['loss', 'time']] = df_scan[['loss', 'time']].astype(float)
                    df_scan.loc[mask, ['loss', 'time']] = [loss_val, elapsed_time]
                    df_scan.to_json(filename, orient='records', indent=2)
                    
                    if verb: 
                        # round before splitting, otherwise 119.97 s would read 1m 60.0s
                        minutes, seconds = divmod(round(elapsed_time, 1), 60)
                        print(f'|{key=}:{value=:.3e} \t| loss={loss_val:.3f}', end='\t')
                        print(f"| completed in {minutes:.0f}m {seconds:.1f}s |")
                else:
                    if verb: 
                        minutes, seconds = divmod(round(df_scan[mask]['time'].values[0], 1), 60)
                        print(f'|{key=}:{value=:.3e} \t | already computed for i_cv={i_cv}, skipping.', end='\t')
                        print(f"| completed in {minutes:.0f}m {seconds:.1f}s |")

    finally:
        # release the lock even after a crash or an interrupt; the placeholder row is then removed by purge_unfinished_scans
        lock_filename.unlink(missing_ok=True)

--------------------------------------------------
scanning key='N_pattern'
--------------------------------------------------
--------------------------------------------------
scanning key='N_time'
--------------------------------------------------
--------------------------------------------------
scanning key='num_delay'
--------------------------------------------------
--------------------------------------------------
scanning key='N_neuron'
--------------------------------------------------
|i_cv=7| 1/13	|key='N_neuron':value=9.200e+01 	| loss=1.000	| completed in 45m 43.3s |
|i_cv=5| 1/13	|key='N_neuron':value=9.200e+01 	| loss=1.000	| completed in 41m 7.1s |
|i_cv=6| 1/13	|key='N_neuron':value=9.200e+01 	| loss=1.000	| completed in 47m 4.0s |
|i_cv=1| 1/13	|key='N_neuron':value=9.200e+01 	| loss=1.000	| completed in 43m 1.5s |
|key='N_neuron':value=9.200e+01 	 | already computed for i_cv=3, skipping.	| completed in 23m 36.4s |
|i_cv=8| 1/13	|key='N_neuron':value=9.200e+01 	| 

In [ ]:
# from matplotlib import ticker
subplotpars_scan = SubplotParams(left=0.125, right=.95, bottom=0.25, top=.975)

for key in scan_dicts:
    print(f"{50*'-'}\nscanning {key=}\n{50*'-'}", )
    filename = data_cache / f'{opt_scan.datetag}_{key}.json'
    lock_filename = filename.with_suffix('.lock')
    if not lock_filename.exists() and filename.exists():
        print(f'Reading {filename=}...')
        df_scan = pd.read_json(filename)
        # keep only the values of the current grid, as older grids may have left other values in the cache
        df_scan = df_scan[np.logical_or.reduce([approx_equals(df_scan[key], value) for value in scan_dicts[key]])]
        df_scan['f1-score'] = 1 - df_scan['loss']
        print(df_scan)
        # https://pandas.pydata.org/pandas-docs/stable/user_guide/visualization.html?highlight=errorbar#visualization-errorbars
        fig, ax = plt.subplots()
        gp_scan = df_scan[[key, 'f1-score']].groupby([key])
        means = gp_scan.mean()
        errors = gp_scan.std()
        q05   = gp_scan.quantile(0.05)
        q95   = gp_scan.quantile(0.95)

        err = np.vstack([means['f1-score'] - q05['f1-score'],
                        q95['f1-score'] - means['f1-score']])

        means['f1-score'].plot.bar(yerr=err, ax=ax, capsize=4, rot=-75,
                                legend=False, alpha=.5)

        # means.plot.bar(yerr=errors, ax=ax, capsize=4, rot=-75, legend=False, alpha=.5)
        ax.set_ylabel(r"$F_1$-score")
        ax.set_xlabel(label_dicts[key])

        # Show only min, median, max ticks with compact labels
        n = len(means)
        # tick_indices = [0, n // 2, n - 1]
        tick_indices = [0, n // 4, n // 2, (3*n) // 4, n - 1]
        all_xticks = ax.get_xticks()
        selected_ticks = all_xticks[:n][tick_indices]
        tick_values = means.index[tick_indices].tolist()
        tick_labels = [str(v) if isinstance(v, bool) else f'{v:.1e}' if isinstance(v, float) else str(int(v)) if isinstance(v, int) else str(v) for v in tick_values]
        ax.set_xticks(selected_ticks)
        ax.set_xticklabels(tick_labels, rotation=-30, ha='left')

        ax.set_ylim(0, 1)
        printfig(fig, 'scan_' + key, fig_width=opt_scan.fig_width/3, fig_height=opt_scan.fig_height, figpath=figpath)
        plt.show()